# Inner Monologue | Reasoning Patterns

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class MonologueState(TypedDict):
    user_message: str
    inner_thoughts: List[str]
    external_response: NotRequired[str]

def think(state: MonologueState) -> dict:
    """Internal reasoning step -- not shown to user."""
    response = model.invoke(
        f"You are an expert support agent. Think through this problem INTERNALLY.\n\n"
        f"User message: {state['user_message']}\n\n"
        f"Think step-by-step:\n"
        f"1. What is the user actually asking?\n"
        f"2. What are the possible causes or interpretations?\n"
        f"3. What information do I need?\n"
        f"4. What is the best approach to help?\n"
        f"5. Are there any risks or edge cases?\n\n"
        f"Write your internal thoughts freely. This will NOT be shown to the user."
    )
    thoughts = list(state.get("inner_thoughts", [])) + [response.content]
    return {"inner_thoughts": thoughts}

def respond(state: MonologueState) -> dict:
    """Generate the external response — explicitly grounded in the thinking output."""
    thoughts = "\n".join(state["inner_thoughts"])
    response = model.invoke(
        f"Based on your internal analysis: {thoughts}\n\n"
        f"Now write your response to the user. Your response must address the key points "
        f"from your analysis above. Specifically:\n"
        f"- Answer the user's core question identified in your analysis\n"
        f"- Address the risks/edge cases you identified\n"
        f"- Use the approach you decided on in your thinking\n\n"
        f"User message: {state['user_message']}\n\n"
        f"Write a helpful, professional response. Do not reveal your internal reasoning process."
    )
    return {"external_response": response.content}

In [5]:
graph = StateGraph(MonologueState)
graph.add_node("think", think)
graph.add_node("respond", respond)

graph.add_edge(START, "think")
graph.add_edge("think", "respond")
graph.add_edge("respond", END)

agent = graph.compile()

In [6]:
# Plot the workflow
plot_mermaid(agent)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	think(think)
	respond(respond)
	__end__([<p>__end__</p>]):::last
	__start__ --> think;
	think --> respond;
	respond --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [7]:
result = agent.invoke({
    "user_message": "My API returns 200 OK but the response body is empty. This started after I updated the SDK.",
    "inner_thoughts": [],
})

# Developer sees both; user sees only external_response
print("=== Inner Thoughts (debug only) ===")
for i, thought in enumerate(result["inner_thoughts"], 1):
    print(f"Thought {i}: {thought[:300]}...")
print(f"\n=== User-Facing Response ===\n{result['external_response']}")

=== Inner Thoughts (debug only) ===
Thought 1: The user's issue is that their API is returning a 200 OK status, which indicates a successful request, but the response body is unexpectedly empty. This started occurring after they updated their SDK.

1. **User's Question**: They are essentially asking why the response body is empty despite getting...

=== User-Facing Response ===
Hello,

I understand that you're experiencing an issue where your API returns a 200 OK status, but the response body is unexpectedly empty after updating your SDK. Let's try to identify and resolve this issue together.

Here are some steps you can take to troubleshoot and potentially resolve the issue:

1. **Review SDK Release Notes**: Check the release notes or documentation for the new SDK version you updated to. Look for any changes related to how API responses are handled, as there might be changes in defaults, processing, or new features that could affect your response.

2. **Inspect the Raw HTTP Response**

In [8]:
stream_invoke(agent, {
    "user_message": "My API returns 200 OK but the response body is empty. This started after I updated the SDK.",
    "inner_thoughts": [],
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'user_message': 'My API returns 200 OK but the response body is empty. This started after I updated the SDK.',
 'inner_thoughts': ["The user's issue is that their API is returning a 200 OK status, which indicates a successful request, but the response body is unexpectedly empty. This started occurring after they updated their SDK.\n\n1. **User's Question**: They are essentially asking why the response body is empty despite getting a 200 OK status after updating their SDK, and how they can fix this issue.\n\n2. **Possible Causes**: \n   - **SDK Changes**: The update might have changed how the response is processed or handled, possibly due to new default settings or a bug.\n   - **Compatibility Issues**: The SDK might not be fully compatible with the current version of the user's API or backend.\n   - **Configuration or Initialization Changes**: The new SDK version might require different configuration or initialization that has not been set correctly.\n   - **API Endpoint Changes**: It